# 00 · Audit of the Kaggle copy of STATS19 (why it was rejected)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam843/uk-road-accident-analytics/blob/main/notebooks/00_kaggle_data_audit.ipynb)

Earlier versions of this project used a Kaggle CSV that claimed to be the UK road accident data
for 2005-2014. This notebook checks that file against the official DfT release and records why
this project uses the **official DfT open data** instead.

**Input:** `DATA_DIR/kaggle/ukroadaccident.csv`. Supply it by hand; this repo never downloads
from Kaggle. If the file is absent, every cell below skips with a message.

Checks:
1. Rows per year: is any year missing?
2. Exact duplicate rows, especially in 2012, compared with the official 2012 total from RAS0101.
3. `Accident_Index` uniqueness, and indexes mangled into scientific notation (e.g. `2.01E+12`) by Excel.
4. Rows with missing or zero coordinates.
5. Codes that do not exist in the DfT data guide.

All numbers come from the cells below.

**Results (this run):**
- 1,504,150 rows built from three concatenated files (2005-2007, 2009-2011, 2012-2014).
  **2008 has 0 rows.**
- 34,155 exact duplicate rows, 34,144 of them in 2012, in one contiguous block. Kaggle 2012 has
  179,715 rows (+23.46% vs RAS0101); after removing the duplicates it is 145,571, exactly the
  official figure. Every other year present matches RAS0101 exactly. The 11 duplicates outside
  2012 all have a scientific-notation index: distinct collisions that became identical once the
  index was corrupted.
- `Accident_Index`: 927,387 unique values for 1,504,150 rows; 569,359 rows (37.9%) are in
  scientific notation (381,416 of them read `2.01E+12`).
- 101 rows have latitude 0 and no longitude. There is no mean-imputation spike.
- 7 fields are text labels instead of codes, mostly from an older code list (for example
  every `light_conditions` value), so they do not match the current data guide.

In [1]:
# Setup: works from the repo root, from notebooks/, or on Google Colab.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("src/config.py").exists():
    if not Path("uk-road-accident-analytics").exists():
        !git clone -q https://github.com/maryam843/uk-road-accident-analytics.git
    %cd uk-road-accident-analytics
    %pip install -q -r requirements.txt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd

from src import clean, config

DATA_DIR = config.get_data_dir()
KAGGLE_PATH = DATA_DIR / "kaggle" / "ukroadaccident.csv"
HAVE_KAGGLE = KAGGLE_PATH.exists()
if HAVE_KAGGLE:
    print(f"Kaggle file: {KAGGLE_PATH.stat().st_size:,} bytes")
else:
    print(f"Skipping audit: {config.display_path(KAGGLE_PATH)} not found. Put the Kaggle CSV there to run this notebook.")

Kaggle file: 449,667,447 bytes


In [3]:
# The Kaggle header spells the road classes "1st_Road_Class" / "2nd_Road_Class".
KAGGLE_RENAMES = {"1st_road_class": "first_road_class", "2nd_road_class": "second_road_class",
                  "1st_road_number": "first_road_number", "2nd_road_number": "second_road_number"}


def load_kaggle(path):
    """Read every column as text so nothing is silently converted."""
    df = pd.read_csv(path, dtype=str, low_memory=False, encoding_errors="replace")
    df = clean.coalesce_duplicates(df.rename(columns=clean.build_column_map(df.columns)))
    return df.rename(columns=KAGGLE_RENAMES)


def kaggle_dates(df):
    """Parse the date column, trying ISO first and then DD/MM/YYYY."""
    iso = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
    return iso.fillna(dmy)


if HAVE_KAGGLE:
    kaggle = load_kaggle(KAGGLE_PATH)
    kaggle["_date"] = kaggle_dates(kaggle)
    year_col = "year" if "year" in kaggle.columns else None
    kaggle["_year"] = (pd.to_numeric(kaggle[year_col], errors="coerce") if year_col
                       else kaggle["_date"].dt.year)
    print(f"{len(kaggle):,} rows, {kaggle.shape[1] - 2} columns")
    print("columns:", list(kaggle.columns[:-2]))
    mismatch = (kaggle["_date"].dt.year != kaggle["_year"]).sum()
    print(f"rows whose Year column disagrees with the year of Date: {int(mismatch):,}")

1,504,150 rows, 33 columns
columns: ['unnamed_0', 'collision_index', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_district', 'local_authority_highway', 'first_road_class', 'first_road_number', 'road_type', 'speed_limit', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing_human_control', 'pedestrian_crossing_physical_facilities', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area', 'did_police_officer_attend_scene_of_collision', 'lsoa_of_collision_location', 'year']
rows whose Year column disagrees with the year of Date: 0


### The unnamed first column is a row counter, not data
The file starts with an unnamed column (`unnamed_0`). If it is kept, no two rows can ever be
exact duplicates, so it is excluded from the duplicate check below. Where the counter
restarts at 0 shows where separate files were concatenated.

In [4]:
if HAVE_KAGGLE:
    counter = pd.to_numeric(kaggle["unnamed_0"])
    restarts = counter.index[counter.diff() < 0].tolist()
    blocks = pd.DataFrame({"first_file_row": [0, *restarts],
                           "last_file_row": [*(r - 1 for r in restarts), len(kaggle) - 1]})
    blocks["rows"] = blocks["last_file_row"] - blocks["first_file_row"] + 1
    blocks["years"] = [f"{int(kaggle['_year'].iloc[a:b + 1].min())}-{int(kaggle['_year'].iloc[a:b + 1].max())}"
                       for a, b in zip(blocks["first_file_row"], blocks["last_file_row"], strict=True)]
    print(f"Row counter restarts {len(restarts)} times -> {len(blocks)} concatenated blocks:")
    display(blocks)

Row counter restarts 2 times -> 3 concatenated blocks:


,first_file_row,last_file_row,rows,years
0,0,570010,570011,2005-2007
1,570011,1039452,469442,2009-2011
2,1039453,1504149,464697,2012-2014


## 1. Rows per year
A year with 0 rows is missing from the file entirely.

In [5]:
if HAVE_KAGGLE:
    years = range(int(kaggle["_year"].min()), int(kaggle["_year"].max()) + 1)
    rows_per_year = kaggle["_year"].value_counts().reindex(years, fill_value=0).rename("rows")
    display(rows_per_year.to_frame().T)
    print("Years with no rows:", [y for y, n in rows_per_year.items() if n == 0])

_year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
rows,198735,189161,182115,0,163554,154414,151474,179715,138660,146322


Years with no rows: [2008]


## 2. Exact duplicate rows
A row is an exact duplicate if every data column (all columns except the row counter) equals
an earlier row.

In [6]:
if HAVE_KAGGLE:
    data_cols = [c for c in kaggle.columns if not c.startswith("_") and c != "unnamed_0"]
    exact_dup = kaggle.duplicated(subset=data_cols, keep="first")
    dup_by_year = (exact_dup.groupby(kaggle["_year"]).sum().reindex(years, fill_value=0)
                   .rename("exact_duplicate_rows"))
    print(f"Exact duplicate rows in the whole file: {int(exact_dup.sum()):,}")
    display(dup_by_year[dup_by_year > 0].to_frame().T)
    if dup_by_year.get(2012, 0):
        pos = exact_dup.to_numpy().nonzero()[0]
        pos_2012 = pos[(kaggle["_year"].to_numpy()[pos] == 2012)]
        print(f"2012 duplicates: {len(pos_2012):,} rows, file rows {pos_2012.min():,}-{pos_2012.max():,} "
              f"({pos_2012.max() - pos_2012.min() + 1:,} rows span)")
        first_copy = kaggle.loc[exact_dup & (kaggle["_year"] == 2012), "collision_index"].head(3).tolist()
        print("examples of duplicated 2012 index values:", first_copy)
    other = exact_dup & (kaggle["_year"] != 2012)
    other_sci = kaggle.loc[other, "collision_index"].str.contains(r"[eE]\+", regex=True)
    print(f"Exact duplicates outside 2012: {int(other.sum())}, of which {int(other_sci.sum())} have an "
          "index in scientific notation (distinct collisions can become identical once the index is lost)")

Exact duplicate rows in the whole file: 34,155


_year,2005,2009,2011,2012,2013,2014
exact_duplicate_rows,3,1,4,34144,1,2


2012 duplicates: 34,144 rows, file rows 1,087,184-1,121,327 (34,144 rows span)
examples of duplicated 2012 index values: ['201201RG40561', '201201RG40562', '201201RG40563']
Exact duplicates outside 2012: 11, of which 11 have an index in scientific notation (distinct collisions can become identical once the index is lost)


### Side by side: Kaggle vs the official pipeline vs RAS0101
`pipeline_rows` are this project's cleaned collisions per year (notebook 01 shows they match
RAS0101 exactly). `kaggle_dedup` removes the exact duplicates found above.

In [7]:
if HAVE_KAGGLE:
    official = clean.load_ras0101(DATA_DIR / "raw" / config.RAS0101_RAW_FILE)["collisions"]
    pipeline = pd.read_parquet(DATA_DIR / "processed" / config.YEARLY_PARQUET)["collisions"]
    side = pd.DataFrame({"kaggle_rows": rows_per_year, "kaggle_exact_dups": dup_by_year})
    side["kaggle_dedup"] = side["kaggle_rows"] - side["kaggle_exact_dups"]
    side["pipeline_rows"] = pipeline.reindex(side.index)
    side["ras0101"] = official.reindex(side.index)
    side["kaggle_minus_ras0101"] = side["kaggle_rows"] - side["ras0101"]
    side["kaggle_dedup_minus_ras0101"] = side["kaggle_dedup"] - side["ras0101"]
    side["pct_kaggle_vs_ras0101"] = 100 * side["kaggle_minus_ras0101"] / side["ras0101"]
    side["pipeline_minus_ras0101"] = side["pipeline_rows"] - side["ras0101"]
    side = side.astype({c: "int64" for c in side.columns if c != "pct_kaggle_vs_ras0101"})
    side.index.name = "year"
    display(side.style.format({c: "{:,}" for c in side.columns if c != "pct_kaggle_vs_ras0101"}
                              | {"pct_kaggle_vs_ras0101": "{:+.2f}%"}))
    total = side.sum(numeric_only=True)
    print(f"Totals {side.index.min()}-{side.index.max()}: Kaggle {int(total['kaggle_rows']):,} rows, "
          f"pipeline {int(total['pipeline_rows']):,}, RAS0101 {int(total['ras0101']):,}")
    print("Years where Kaggle differs from RAS0101 by more than 0.5%:",
          side.index[side["pct_kaggle_vs_ras0101"].abs() > 0.5].tolist())

,kaggle_rows,kaggle_exact_dups,kaggle_dedup,pipeline_rows,ras0101,kaggle_minus_ras0101,kaggle_dedup_minus_ras0101,pct_kaggle_vs_ras0101,pipeline_minus_ras0101
year,,,,,,,,,
2005,"198,735",3,"198,732","198,735","198,735",0,-3,+0.00%,0
2006,"189,161",0,"189,161","189,161","189,161",0,0,+0.00%,0
2007,"182,115",0,"182,115","182,115","182,115",0,0,+0.00%,0
2008,0,0,0,"170,591","170,591","-170,591","-170,591",-100.00%,0
2009,"163,554",1,"163,553","163,554","163,554",0,-1,+0.00%,0
2010,"154,414",0,"154,414","154,414","154,414",0,0,+0.00%,0
2011,"151,474",4,"151,470","151,474","151,474",0,-4,+0.00%,0
2012,"179,715","34,144","145,571","145,571","145,571","34,144",0,+23.46%,0
2013,"138,660",1,"138,659","138,660","138,660",0,-1,+0.00%,0


Totals 2005-2014: Kaggle 1,504,150 rows, pipeline 1,640,597, RAS0101 1,640,597
Years where Kaggle differs from RAS0101 by more than 0.5%: [2008, 2012]


In [8]:
# Result file for src/dashboard_exports.py: Kaggle rows per year from the table above.
if HAVE_KAGGLE:
    config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    rows_path = config.RESULTS_DIR / "kaggle_rows_by_year.csv"
    side[["kaggle_rows"]].to_csv(rows_path)
    print("Wrote", config.display_path(rows_path))

Wrote reports/results/kaggle_rows_by_year.csv


## 3. `Accident_Index` uniqueness and scientific notation

In [9]:
if HAVE_KAGGLE:
    idx = kaggle["collision_index"].astype(str)
    sci = idx.str.fullmatch(r"\d+(\.\d+)?[eE][+-]?\d+")
    print(f"rows: {len(idx):,}")
    print(f"unique index values: {idx.nunique():,}")
    print(f"rows whose index is in scientific notation: {int(sci.sum()):,} ({sci.mean():.1%})")
    print(f"distinct scientific-notation values: {idx[sci].nunique():,}; examples:",
          idx[sci].value_counts().head(5).to_dict())
    print(f"unique index values among rows NOT in scientific notation: {idx[~sci].nunique():,} "
          f"of {int((~sci).sum()):,}")
    display(pd.DataFrame({"rows": kaggle["_year"].value_counts().sort_index(),
                          "sci_notation_rows": sci.groupby(kaggle["_year"]).sum(),
                          "share": sci.groupby(kaggle["_year"]).mean()}).T)

rows: 1,504,150


unique index values: 927,387
rows whose index is in scientific notation: 569,359 (37.9%)


distinct scientific-notation values: 15,395; examples: {'2.01E+12': 381416, '2.00913E+12': 6255, '2.01013E+12': 5760, '2.01113E+12': 5402, '2.00946E+12': 5158}


unique index values among rows NOT in scientific notation: 911,992 of 934,791


_year,2005,2006,2007,2009,2010,2011,2012,2013,2014
rows,198735.000000,189161.000000,182115.000000,163554.000000,154414.000000,151474.000000,179715.000000,138660.000000,146322.000000
sci_notation_rows,73300.000000,70252.000000,68251.000000,62443.000000,56074.000000,56805.000000,68155.000000,57585.000000,56494.000000
share,0.368833,0.371387,0.374769,0.381788,0.363141,0.375015,0.379239,0.415296,0.386094


## 4. Missing or zero coordinates

In [10]:
if HAVE_KAGGLE:
    lat = pd.to_numeric(kaggle.get("latitude"), errors="coerce")
    lon = pd.to_numeric(kaggle.get("longitude"), errors="coerce")
    print(f"missing latitude or longitude: {int((lat.isna() | lon.isna()).sum()):,}")
    print(f"latitude exactly 0: {int((lat == 0).sum()):,}; longitude exactly 0: {int((lon == 0).sum()):,}")
    print(f"rows with latitude 0 AND longitude missing: {int(((lat == 0) & lon.isna()).sum()):,}")
    outside = ~(lat.between(*config.GB_LAT_RANGE) & lon.between(*config.GB_LON_RANGE))
    print(f"outside the GB bounding box (incl. missing): {int(outside.sum()):,}")
    filled_mean = lat.round(4).value_counts().head(3)
    print("most frequent latitude values (a mean-imputed value would show up as a spike):")
    display(filled_mean.to_frame("rows"))

missing latitude or longitude: 101
latitude exactly 0: 101; longitude exactly 0: 0
rows with latitude 0 AND longitude missing: 101
outside the GB bounding box (incl. missing): 101
most frequent latitude values (a mean-imputed value would show up as a spike):


,rows
latitude,
51.5110,370
51.5198,331
51.5141,331


## 5. Codes not in the DfT data guide
Needs `DATA_DIR/raw/data_guide.xlsx`. Current and `_historic` code lists are both accepted.
The Kaggle file stores some fields as numeric codes and others as text labels, so both are
checked: numbers against the guide's codes, text against the guide's labels (ignoring case).

In [11]:
guide_path = DATA_DIR / "raw" / config.DATA_GUIDE_RAW_FILE
if HAVE_KAGGLE and guide_path.exists():
    codes = clean.load_code_list(guide_path)
    rows = []
    for col in clean.COLLISION_CODED_FIELDS:
        if col not in kaggle.columns:
            continue
        fields = [col, f"{col}_historic", col.removesuffix("_historic")]
        guide = codes.loc[codes["field"].isin(fields)]
        allowed_codes = set(guide["code"])
        allowed_labels = set(guide["label"].dropna().str.lower())
        raw = kaggle[col]
        values = pd.to_numeric(raw, errors="coerce")
        is_text = raw.notna() & values.isna()
        odd_code = values.notna() & ~values.isin(allowed_codes)
        odd_text = is_text & ~raw.str.strip().str.lower().isin(allowed_labels)
        rows.append({"column": col, "numeric_rows": int(values.notna().sum()),
                     "codes_not_in_guide": int(odd_code.sum()),
                     "text_rows": int(is_text.sum()), "text_not_in_guide": int(odd_text.sum()),
                     "missing": int(raw.isna().sum()),
                     "examples": (sorted(values[odd_code].unique().tolist())[:4]
                                  + raw[odd_text].value_counts().index[:3].tolist())})
    odd_codes = pd.DataFrame(rows).set_index("column")
    display(odd_codes)
    print(f"Fields stored as text labels: {odd_codes.index[odd_codes['text_rows'] > 0].tolist()}")
    print(f"Rows with a code or label not in the data guide, summed over fields: "
          f"{int(odd_codes['codes_not_in_guide'].sum() + odd_codes['text_not_in_guide'].sum()):,}")
elif HAVE_KAGGLE:
    print("Data guide not found; run `python -m src.download --only guide` first.")

,numeric_rows,codes_not_in_guide,text_rows,text_not_in_guide,missing,examples
column,,,,,,
police_force,1504150,0,0,0,0,[]
collision_severity,1504150,0,0,0,0,[]
day_of_week,1504150,0,0,0,0,[]
first_road_class,1504150,0,0,0,0,[]
road_type,0,0,1504150,0,0,[]
junction_control,0,0,901315,889657,602835,"[Giveway or uncontrolled, Automatic traffic si..."
second_road_class,1504150,0,0,0,0,[]
light_conditions,0,0,1504150,1504150,0,"[Daylight: Street light present, Darkness: Str..."
weather_conditions,0,0,1504150,1434035,0,"[Fine without high winds, Raining without high..."


Fields stored as text labels: ['road_type', 'junction_control', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'did_police_officer_attend_scene_of_collision']
Rows with a code or label not in the data guide, summed over fields: 4,297,910


## Why this file was rejected

Each failure listed here is checked by a cell above. The size of each problem is whatever
those cells print.

- **Missing year:** a year with 0 rows breaks any trend, seasonality or forecasting work.
- **Duplicated rows:** exact copies inflate one year's count above the official total, which
  would show up as a fake spike in a forecast. They also put the same collision in both train
  and test.
- **Corrupted index:** values stored in scientific notation are no longer unique, so
  collisions cannot be joined to casualties, and duplicates cannot be found by index.
- **Missing coordinates:** rows without coordinates cannot be mapped; any pipeline that fills
  them with a mean or 0 puts collisions in the sea or at one fake point.
- **Mixed codes and labels:** some fields are numeric codes and others are text labels from an
  older code list, so they cannot be decoded consistently against the current data guide.

The official DfT release (`src/download.py`) has none of these problems by construction,
can be reconciled with published totals (notebook 01), and is licensed under OGL v3.0.